In [1]:
from google.colab import files
import io
import pandas as pd

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

Saving spotify_recommendations.csv to spotify_recommendations.csv


In [2]:
"""
K-Nearest Neighbors (KNN)
Dataset: spotify_recommendations.csv
Predicts: Whether a user will like a song (binary: 1 = liked, 0 = not liked)
Features: Audio features (danceability, energy, tempo, valence, etc.)
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

#1. Load data
df = pd.read_csv("spotify_recommendations.csv")
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['liked'].value_counts()}\n")

#2. Feature selection
# Audio features — all continuous numeric, ideal for distance-based KNN
# Exclude duration_ms, key, mode, time_signature to keep feature space clean
AUDIO_FEATURES = [
    "danceability",      # how suitable for dancing (0–1)
    "energy",            # perceptual intensity and activity (0–1)
    "loudness",          # overall loudness in dB
    "speechiness",       # presence of spoken words (0–1)
    "acousticness",      # confidence track is acoustic (0–1)
    "instrumentalness",  # predicts whether track has no vocals (0–1)
    "liveness",          # presence of audience (0–1)
    "valence",           # musical positiveness (0–1)
    "tempo",             # BPM
]

X = df[AUDIO_FEATURES]
y = df["liked"]

#3. Train/Validation/Test split (70/15/15)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp
    # 0.176 of 0.85 ≈ 0.15 of total
)
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

#4. Scale features
# KNN is distance-based — scaling is critical so tempo (0–250) doesn't dominate loudness (−60 to 0) or 0–1 features.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

#5. Tune K on validation set
k_range = range(1, 31)
val_f1_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k, metric="euclidean")
    knn.fit(X_train_scaled, y_train)
    preds = knn.predict(X_val_scaled)
    val_f1_scores.append(f1_score(y_val, preds, zero_division=0))

best_k = k_range[np.argmax(val_f1_scores)]
best_val_f1 = max(val_f1_scores)
print(f"\nBest K: {best_k}  (Validation F1 = {best_val_f1:.4f})")

#Plot K-tuning curve
plt.figure(figsize=(9, 4))
plt.plot(list(k_range), val_f1_scores, marker="o", color="#1DB954", linewidth=2)
plt.axvline(best_k, linestyle="--", color="#191414", alpha=0.6, label=f"Best K={best_k}")
plt.xlabel("K (number of neighbors)")
plt.ylabel("Validation F1 Score")
plt.title("KNN — Validation F1 vs K")
plt.legend()
plt.tight_layout()
plt.savefig("model1_k_tuning.png", dpi=150)
plt.close()
print("Saved: model1_k_tuning.png")

#6. Final evaluation on test set
knn_final = KNeighborsClassifier(n_neighbors=best_k, metric="euclidean")
knn_final.fit(X_train_scaled, y_train)
y_pred = knn_final.predict(X_test_scaled)

print("\n TEST SET RESULTS ")
print(classification_report(y_test, y_pred, target_names=["Not Liked", "Liked"]))

test_f1        = f1_score(y_test, y_pred, zero_division=0)
test_precision = precision_score(y_test, y_pred, zero_division=0)
test_recall    = recall_score(y_test, y_pred, zero_division=0)
print(f"Precision: {test_precision:.4f} | Recall: {test_recall:.4f} | F1: {test_f1:.4f}")

#Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=["Not Liked", "Liked"],
            yticklabels=["Not Liked", "Liked"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"KNN Confusion Matrix (K={best_k})")
plt.tight_layout()
plt.savefig("model1_confusion_matrix.png", dpi=150)
plt.close()
print("Saved: model1_confusion_matrix.png")

#7. Feature importance
# KNN has no built-in feature importance, so use permutation importance: shuffle each feature and measure how much F1 drops
baseline_f1 = f1_score(y_test, knn_final.predict(X_test_scaled), zero_division=0)
importance = {}

for i, feat in enumerate(AUDIO_FEATURES):
    X_permuted = X_test_scaled.copy()
    np.random.seed(42)
    X_permuted[:, i] = np.random.permutation(X_permuted[:, i])
    perm_preds = knn_final.predict(X_permuted)
    drop = baseline_f1 - f1_score(y_test, perm_preds, zero_division=0)
    importance[feat] = drop

importance_df = pd.DataFrame.from_dict(importance, orient="index", columns=["F1 Drop"])
importance_df = importance_df.sort_values("F1 Drop", ascending=True)

plt.figure(figsize=(8, 5))
importance_df["F1 Drop"].plot(kind="barh", color="#1DB954", edgecolor="white")
plt.xlabel("Drop in F1 when feature is permuted")
plt.title("KNN — Permutation Feature Importance")
plt.tight_layout()
plt.savefig("model1_feature_importance.png", dpi=150)
plt.close()
print("Saved: model1_feature_importance.png")

#8. Recommendation function
def recommend_songs_knn(liked_song_features: dict, song_pool: pd.DataFrame,
                         scaler, knn_model, top_n: int = 10) -> pd.DataFrame:
    """
    Given the audio features of songs a user liked, find the most similar
    songs from the pool and return those the KNN predicts the user will enjoy.

    Args:
        liked_song_features: dict mapping feature names to values for a liked song
        song_pool:           DataFrame of candidate songs (must include AUDIO_FEATURES)
        scaler:              Fitted StandardScaler
        knn_model:           Fitted KNeighborsClassifier
        top_n:               Number of recommendations to return

    Returns:
        DataFrame of top_n recommended songs with their predicted distances.
    """
    query = pd.DataFrame([liked_song_features])[AUDIO_FEATURES]
    query_scaled = scaler.transform(query)

    pool_scaled = scaler.transform(song_pool[AUDIO_FEATURES])
    distances, indices = knn_model.kneighbors(query_scaled, n_neighbors=min(top_n, len(song_pool)))

    recommendations = song_pool.iloc[indices[0]].copy()
    recommendations["distance"] = distances[0]
    return recommendations.sort_values("distance")

# Demo: find the most similar songs to a liked track (first liked song in test set)
liked_idx = y_test[y_test == 1].index[0]
liked_features = dict(X_test.loc[liked_idx])
pool = df[AUDIO_FEATURES].drop(index=liked_idx, errors="ignore")
recs = recommend_songs_knn(liked_features, pool, scaler, knn_final)
print("\n── SAMPLE RECOMMENDATIONS (top 5) ──")
print(recs[AUDIO_FEATURES + ["distance"]].head(5).to_string())

Dataset shape: (195, 14)
Class balance:
liked
1    100
0     95
Name: count, dtype: int64

Train: 135 | Val: 30 | Test: 30

Best K: 21  (Validation F1 = 0.9333)
Saved: model1_k_tuning.png

 TEST SET RESULTS 
              precision    recall  f1-score   support

   Not Liked       0.86      0.80      0.83        15
       Liked       0.81      0.87      0.84        15

    accuracy                           0.83        30
   macro avg       0.83      0.83      0.83        30
weighted avg       0.83      0.83      0.83        30

Precision: 0.8125 | Recall: 0.8667 | F1: 0.8387
Saved: model1_confusion_matrix.png
Saved: model1_feature_importance.png

── SAMPLE RECOMMENDATIONS (top 5) ──
     danceability  energy  loudness  speechiness  acousticness  instrumentalness  liveness  valence    tempo  distance
31          0.668  0.4590   -12.072       0.1180        0.0499          0.000001    0.4080    0.525  159.021  1.416162
35          0.307  0.0515   -28.493       0.0324        0.7080       

In [3]:
from google.colab import files
files.download("model1_confusion_matrix.png")
files.download("model1_k_tuning.png")
files.download("model1_feature_importance.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>